In [214]:
import os
import pandas as pd

import re
import html
import emoji
import unicodedata

## Setup
In this section, we set up the environment by importing necessary libraries, loading configuration variables, and reading the main dataset into a pandas DataFrame for further analysis.

In [215]:
from dotenv import load_dotenv
load_dotenv()

True

In [216]:
# Load paths from .env file
data_dir = os.getenv("DATA_DIR")

# Load the unified TSV file with image hashes
annotations_file = os.path.join(data_dir, "annotations_with_hash.tsv")

In [217]:
df = pd.read_csv(annotations_file, sep="\t")

In [218]:
df.head()

,tweet_id,image_id,text_info,text_info_conf,image_info,image_info_conf,text_human,text_human_conf,image_human,image_human_conf,...,image_damage_conf,tweet_text,image_url,image_path,disaster_type,general_disaster_type,img_width,img_height,img_aspect_ratio,img_hash_str
0,917791044158185473,917791044158185473_0,informative,1.0000,informative,0.6766,other_relevant_information,1.0000,other_relevant_information,0.6766,...,NaN,RT @Gizmodo: Wildfires raging through Northern California are terrifying https://t.co/dI73RFzX2i https://t.co/k4KnvIimsU,http://pbs.twimg.com/media/DLyi_WYVYAApwNg.jpg,data_image/california_wildfires/10_10_2017/917791044158185473_0.jpg,california_wildfires,wildfire,800,450,1.777778,86c476bb39c2b16c
1,917791130590183424,917791130590183424_0,informative,1.0000,informative,0.6667,infrastructure_and_utility_damage,1.0000,affected_individuals,0.6667,...,NaN,PHOTOS: Deadly wildfires rage in California https://t.co/td9xT3vXOL https://t.co/OimwAncLew,http://pbs.twimg.com/media/DLymKm9UMAAu0qw.jpg,data_image/california_wildfires/10_10_2017/917791130590183424_0.jpg,california_wildfires,wildfire,1200,677,1.772526,f28e9aa98b96a730
2,917791291823591425,917791291823591425_0,informative,0.6813,informative,1.0000,other_relevant_information,0.6813,infrastructure_and_utility_damage,1.0000,...,1.0,"RT @Cal_OES: PLS SHARE: We're capturing wildfire response, recovery info here: https://t.co/r89LKpjLPj https://t.co/HiA1oQF2Ax",http://pbs.twimg.com/media/DLudaaZV4AAjT7x.jpg,data_image/california_wildfires/10_10_2017/917791291823591425_0.jpg,california_wildfires,wildfire,640,480,1.333333,a9bdb18391838bba
3,917791291823591425,917791291823591425_1,informative,0.6813,not_informative,1.0000,other_relevant_information,0.6813,not_humanitarian,1.0000,...,NaN,"RT @Cal_OES: PLS SHARE: We're capturing wildfire response, recovery info here: https://t.co/r89LKpjLPj https://t.co/HiA1oQF2Ax",http://pbs.twimg.com/media/DLudaZXUMAABAEZ.jpg,data_image/california_wildfires/10_10_2017/917791291823591425_1.jpg,california_wildfires,wildfire,1200,900,1.333333,b582566fab45cac8
4,917792092100988929,917792092100988929_0,informative,0.6727,informative,0.6612,other_relevant_information,0.6727,infrastructure_and_utility_damage,0.6612,...,1.0,RT @TIME: California's raging wildfires as you've never seen them before https://t.co/OksQOZ2LHH https://t.co/oHTMbrM2Jx,http://pbs.twimg.com/media/DLwNe-NXUAE0XCw.jpg,data_image/california_wildfires/10_10_2017/917792092100988929_0.jpg,california_wildfires,wildfire,600,400,1.500000,86826be7394ed43a


## Duplicated tweet_id

In [219]:
# Show a few examples of duplicated tweet_id entries
print("Sample duplicated tweet_id entries (same tweet, different images):")
sample_tweet_dups = df[df["tweet_id"].duplicated(keep=False)].sort_values("tweet_id").head(10)
display(sample_tweet_dups[["tweet_id", "image_id", "tweet_text", "text_info", "text_human"]])


Sample duplicated tweet_id entries (same tweet, different images):


,tweet_id,image_id,tweet_text,text_info,text_human
17072,869972354004393987,869972354004393987_2,Pak Navy continues Humanitarian Assistance and Disaster Relief Operations in Flood Stricken Sri Lanka. RESPECT https://t.co/LzDAR9i17Z,informative,rescue_volunteering_or_donation_effort
17073,869972354004393987,869972354004393987_3,Pak Navy continues Humanitarian Assistance and Disaster Relief Operations in Flood Stricken Sri Lanka. RESPECT https://t.co/LzDAR9i17Z,informative,rescue_volunteering_or_donation_effort
17070,869972354004393987,869972354004393987_0,Pak Navy continues Humanitarian Assistance and Disaster Relief Operations in Flood Stricken Sri Lanka. RESPECT https://t.co/LzDAR9i17Z,informative,rescue_volunteering_or_donation_effort
17071,869972354004393987,869972354004393987_1,Pak Navy continues Humanitarian Assistance and Disaster Relief Operations in Flood Stricken Sri Lanka. RESPECT https://t.co/LzDAR9i17Z,informative,rescue_volunteering_or_donation_effort
17074,869977622377320448,869977622377320448_0,"RT @DisastersChart: #Sentinel2 was used to map the #flood in Matara, #SriLanka, on 28 May: https://t.co/LCo1VVKgEZ https://t.co/f9o3enK8ka",informative,other_relevant_information
17076,869977622377320448,869977622377320448_2,"RT @DisastersChart: #Sentinel2 was used to map the #flood in Matara, #SriLanka, on 28 May: https://t.co/LCo1VVKgEZ https://t.co/f9o3enK8ka",informative,other_relevant_information
17075,869977622377320448,869977622377320448_1,"RT @DisastersChart: #Sentinel2 was used to map the #flood in Matara, #SriLanka, on 28 May: https://t.co/LCo1VVKgEZ https://t.co/f9o3enK8ka",informative,other_relevant_information
17093,870008928054259712,870008928054259712_0,"@Anchoveebrother Hello dear Chile birds, Martin fisherman, Mora eagle, Pequen..✌ἿD❤️ https://t.co/tN4qxsUHYf",not_informative,not_humanitarian
17095,870008928054259712,870008928054259712_2,"@Anchoveebrother Hello dear Chile birds, Martin fisherman, Mora eagle, Pequen..✌ἿD❤️ https://t.co/tN4qxsUHYf",not_informative,not_humanitarian
17094,870008928054259712,870008928054259712_1,"@Anchoveebrother Hello dear Chile birds, Martin fisherman, Mora eagle, Pequen..✌ἿD❤️ https://t.co/tN4qxsUHYf",not_informative,not_humanitarian


In [220]:
duplicated_tweets = df[df.duplicated(subset=["tweet_id"], keep=False)].copy()

# Find tweet_id groups with inconsistent text_info or text_human
conflicting_tweet_ids = (
	duplicated_tweets.groupby("tweet_id")[["text_info", "text_human"]]
	.nunique()
	.query("text_info > 1 or text_human > 1")
	.index
)

print(f"⚠️ Tweet IDs with conflicting text labels: {len(conflicting_tweet_ids)}")

# Show a few examples
df_conflicting_tweets = df[df["tweet_id"].isin(conflicting_tweet_ids)].sort_values("tweet_id")
display(df_conflicting_tweets[["tweet_id", "tweet_text", "text_info", "text_human", "image_path"]].head(10))


⚠️ Tweet IDs with conflicting text labels: 40


,tweet_id,tweet_text,text_info,text_human,image_path
17492,872235889820291072,"Accumulating hail in Cleveland, NM [Mora Co.]...Courtesy: Geri Roper #nmwx https://t.co/XuvOel2cN4",informative,other_relevant_information,data_image/srilanka_floods/6_6_2017/872235889820291072_1.jpg
17491,872235889820291072,"Accumulating hail in Cleveland, NM [Mora Co.]...Courtesy: Geri Roper #nmwx https://t.co/XuvOel2cN4",not_informative,not_humanitarian,data_image/srilanka_floods/6_6_2017/872235889820291072_0.jpg
17550,872672448394780672,#Srilanka #Floods Seen From #Space https://t.co/5y7jtvE2Yk,not_informative,not_humanitarian,data_image/srilanka_floods/8_6_2017/872672448394780672_1.jpg
17549,872672448394780672,#Srilanka #Floods Seen From #Space https://t.co/5y7jtvE2Yk,informative,other_relevant_information,data_image/srilanka_floods/8_6_2017/872672448394780672_0.jpg
1595,901646123080830976,RT @yIIeza: When we get back to SCHS after Harvey hits : https://t.co/kHMnnURUAA,not_informative,not_humanitarian,data_image/hurricane_harvey/27_8_2017/901646123080830976_1.jpg
1596,901646123080830976,RT @yIIeza: When we get back to SCHS after Harvey hits : https://t.co/kHMnnURUAA,not_informative,not_humanitarian,data_image/hurricane_harvey/27_8_2017/901646123080830976_2.jpg
1597,901646123080830976,RT @yIIeza: When we get back to SCHS after Harvey hits : https://t.co/kHMnnURUAA,informative,other_relevant_information,data_image/hurricane_harvey/27_8_2017/901646123080830976_3.jpg
1594,901646123080830976,RT @yIIeza: When we get back to SCHS after Harvey hits : https://t.co/kHMnnURUAA,not_informative,not_humanitarian,data_image/hurricane_harvey/27_8_2017/901646123080830976_0.jpg
3431,905491119705702400,"PHOTOS: New album ""Texas recovers from Harvey"" posted on our Facebook page https://t.co/h98YQdjIv7 (CNS/Bob Roller) https://t.co/ybiqlVD2mi",informative,other_relevant_information,data_image/hurricane_harvey/6_9_2017/905491119705702400_1.jpg
3432,905491119705702400,"PHOTOS: New album ""Texas recovers from Harvey"" posted on our Facebook page https://t.co/h98YQdjIv7 (CNS/Bob Roller) https://t.co/ybiqlVD2mi",informative,other_relevant_information,data_image/hurricane_harvey/6_9_2017/905491119705702400_2.jpg


In [221]:
# Step 1.1 - Handle tweet_id duplicates (same tweet + different images)
def reconcile_labels(group, col):
	return group.mode().iloc[0] if not group.mode().empty else group.iloc[0]

# Apply reconciliation only to duplicates using mapping
for col in ["text_info", "text_human"]:
	reconciled = duplicated_tweets.groupby("tweet_id")[col].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0])
	df.loc[df["tweet_id"].isin(reconciled.index), col] = df["tweet_id"].map(reconciled)

# Mark these rows to go to test set
df["force_test"] = df["tweet_id"].isin(duplicated_tweets["tweet_id"])  # bool flag

In [222]:
# Tweet duplication stats
n_total = len(df)
n_dup_tweets = df["tweet_id"].duplicated(keep=False).sum()
n_test_tweet_entries = df[df["tweet_id"].isin(duplicated_tweets["tweet_id"])].shape[0]
n_unique_dup_tweet_ids = duplicated_tweets["tweet_id"].nunique()

print(f"Total duplicated tweet_id rows: {n_dup_tweets}")
print(f"Entries added to test set due to tweet duplication: {n_test_tweet_entries}")
print(f"Unique tweet_id groups reconciled: {n_unique_dup_tweet_ids}")


Total duplicated tweet_id rows: 3169
Entries added to test set due to tweet duplication: 3169
Unique tweet_id groups reconciled: 1145


### Duplicated images

In [223]:
# Show a few examples of duplicated images used in different tweets
print("Sample duplicated image hash entries (same image in different tweets):")
sample_img_dups = df[df["img_hash_str"].duplicated(keep=False)].sort_values("img_hash_str").head(10)
display(sample_img_dups[["img_hash_str", "tweet_id", "image_path", "image_info", "image_human", "image_damage"]])

Sample duplicated image hash entries (same image in different tweets):


,img_hash_str,tweet_id,image_path,image_info,image_human,image_damage
5840,8000000000000000,909777730924896256,data_image/hurricane_harvey/18_9_2017/909777730924896256_0.jpg,not_informative,not_humanitarian,NaN
8490,8000000000000000,909839409222230017,data_image/hurricane_irma/18_9_2017/909839409222230017_0.jpg,not_informative,not_humanitarian,NaN
10102,8000000000000000,910197668680540160,data_image/hurricane_irma/19_9_2017/910197668680540160_0.jpg,not_informative,not_humanitarian,NaN
5289,802b2deaca2e5e7a,908168392716234753,data_image/hurricane_harvey/14_9_2017/908168392716234753_0.jpg,informative,infrastructure_and_utility_damage,little_or_no_damage
3823,802b2deaca2e5e7a,905900370412441600,data_image/hurricane_harvey/7_9_2017/905900370412441600_0.jpg,informative,infrastructure_and_utility_damage,mild_damage
1919,8072fea0985a5ed7,901814144680255488,data_image/hurricane_harvey/27_8_2017/901814144680255488_0.jpg,informative,infrastructure_and_utility_damage,severe_damage
1929,8072fea0985a5ed7,901822511750406144,data_image/hurricane_harvey/27_8_2017/901822511750406144_0.jpg,informative,infrastructure_and_utility_damage,severe_damage
11934,80734de64ac55bad,913176642889109504,data_image/hurricane_maria/27_9_2017/913176642889109504_0.jpg,informative,infrastructure_and_utility_damage,mild_damage
11904,80734de64ac55bad,913129211417829378,data_image/hurricane_maria/27_9_2017/913129211417829378_0.jpg,informative,affected_individuals,NaN
15496,80b4be4b5656956d,930457005550112770,data_image/iraq_iran_earthquake/14_11_2017/930457005550112770_0.jpg,informative,infrastructure_and_utility_damage,severe_damage


In [224]:
hash_counts = df["img_hash_str"].value_counts()
duplicate_hashes = hash_counts[hash_counts > 1].index

# Find img_hash groups with inconsistent image labels
conflicting_img_hashes = (
	df[df["img_hash_str"].isin(duplicate_hashes)]
	.groupby("img_hash_str")[["image_info", "image_human", "image_damage"]]
	.nunique()
	.query("image_info > 1 or image_human > 1 or image_damage > 1")
	.index
)

print(f"⚠️ Image hashes with conflicting image labels: {len(conflicting_img_hashes)}")

# Show a few examples
df_conflicting_images = df[df["img_hash_str"].isin(conflicting_img_hashes)].sort_values("img_hash_str")
display(df_conflicting_images[["img_hash_str", "tweet_id", "image_path", "image_info", "image_human", "image_damage"]].head(10))


⚠️ Image hashes with conflicting image labels: 126


,img_hash_str,tweet_id,image_path,image_info,image_human,image_damage
3823,802b2deaca2e5e7a,905900370412441600,data_image/hurricane_harvey/7_9_2017/905900370412441600_0.jpg,informative,infrastructure_and_utility_damage,mild_damage
5289,802b2deaca2e5e7a,908168392716234753,data_image/hurricane_harvey/14_9_2017/908168392716234753_0.jpg,informative,infrastructure_and_utility_damage,little_or_no_damage
11904,80734de64ac55bad,913129211417829378,data_image/hurricane_maria/27_9_2017/913129211417829378_0.jpg,informative,affected_individuals,NaN
11934,80734de64ac55bad,913176642889109504,data_image/hurricane_maria/27_9_2017/913176642889109504_0.jpg,informative,infrastructure_and_utility_damage,mild_damage
11382,80d47f43786a26eb,911982031898365952,data_image/hurricane_maria/24_9_2017/911982031898365952_0.jpg,informative,other_relevant_information,NaN
12508,80d47f43786a26eb,914886154402615296,data_image/hurricane_maria/2_10_2017/914886154402615296_0.jpg,not_informative,not_humanitarian,NaN
2181,81fde0865f4af0d8,904339363072544768,data_image/hurricane_harvey/3_9_2017/904339363072544768_0.jpg,informative,vehicle_damage,NaN
4332,81fde0865f4af0d8,906687728674332672,data_image/hurricane_harvey/10_9_2017/906687728674332672_0.jpg,informative,infrastructure_and_utility_damage,severe_damage
17489,82ade3f4c574711a,872175891929063426,data_image/srilanka_floods/6_6_2017/872175891929063426_0.jpg,informative,affected_individuals,NaN
17186,82ade3f4c574711a,870209884452552705,data_image/srilanka_floods/1_6_2017/870209884452552705_0.jpg,informative,infrastructure_and_utility_damage,severe_damage


In [225]:
# Step 1.2 - Handle image duplicates using precomputed hash
for col in ["image_info", "image_human", "image_damage"]:
	df.loc[df["img_hash_str"].isin(duplicate_hashes), col] = (
		df.groupby("img_hash_str")[col].transform(lambda g: g.mode().iloc[0] if not g.mode().empty else g.iloc[0])
	)

# Mark image duplicates to go to test set
df.loc[df["img_hash_str"].isin(duplicate_hashes), "force_test"] = True

In [226]:
# Image hash duplication stats
n_image_dups = df["img_hash_str"].duplicated(keep=False).sum()
n_test_image_entries = df[df["img_hash_str"].isin(duplicate_hashes)].shape[0]
n_unique_img_hash_groups = len(duplicate_hashes)

print(f"Total duplicated image hash rows: {n_image_dups}")
print(f"Entries added to test set due to image duplication: {n_test_image_entries}")
print(f"Unique image hash groups reconciled: {n_unique_img_hash_groups}")

Total duplicated image hash rows: 1222
Entries added to test set due to image duplication: 1222
Unique image hash groups reconciled: 494


In [227]:
# Overall test set stats
n_force_test = df["force_test"].sum()
pct_force_test = n_force_test / n_total * 100

print(f"Total entries marked for test set (force_test=True): {n_force_test} ({pct_force_test:.2f}%)")


Total entries marked for test set (force_test=True): 4247 (23.49%)


## Fusion Labels

In [228]:
def fuse_info(row):
	if row["text_info"] == "informative" or row["image_info"] == "informative":
		return "informative"
	return "not_informative"

df["mm_info"] = df.apply(fuse_info, axis=1)

In [229]:
def fuse_human(row):
	t, i = row["text_human"], row["image_human"]
	
	if t==i:
		return t
	if str(t).lower() == "not_humanitatian":
		return i
	if str(i).lower() == "not_humanitatian":
		return t
	return "conflict"  # if they disagree
df["mm_human"] = df.apply(fuse_human, axis=1)

In [230]:
print("mm_info distribution:")
print(df["mm_info"].value_counts(), "\n")

print("mm_human distribution:")
print(df["mm_human"].value_counts())

mm_info distribution:
mm_info
informative        13831
not_informative     4251
Name: count, dtype: int64 

mm_human distribution:
mm_human
conflict                                  10006
not_humanitarian                           4297
other_relevant_information                 1736
rescue_volunteering_or_donation_effort     1177
infrastructure_and_utility_damage           754
affected_individuals                         63
injured_or_dead_people                       25
vehicle_damage                               22
missing_or_found_people                       2
Name: count, dtype: int64


In [236]:
df[["mm_human"]=="conflict"].head(10)


KeyError: False

## Text preprocessing



In [231]:
pd.set_option('display.max_colwidth', None)
df["tweet_text"].head(10)

0             RT @Gizmodo: Wildfires raging through Northern California are terrifying https://t.co/dI73RFzX2i https://t.co/k4KnvIimsU
1                                          PHOTOS: Deadly wildfires rage in California https://t.co/td9xT3vXOL https://t.co/OimwAncLew
2       RT @Cal_OES: PLS SHARE: We're capturing wildfire response, recovery info here: https://t.co/r89LKpjLPj https://t.co/HiA1oQF2Ax
3       RT @Cal_OES: PLS SHARE: We're capturing wildfire response, recovery info here: https://t.co/r89LKpjLPj https://t.co/HiA1oQF2Ax
4             RT @TIME: California's raging wildfires as you've never seen them before https://t.co/OksQOZ2LHH https://t.co/oHTMbrM2Jx
5                         Wildfires Threaten California's First Legal Cannabis Harvest https://t.co/BSuAdfwN62 https://t.co/wNqfLrNmTp
6    Mass Evacuations in California as Wildfires Kill at Least 10 https://t.co/gyoKFWZuMB #CaliforniaWildfires https://t.co/KEFtjITetK
7        RT @KAKEnews: California wildfires destroy mor

In [232]:
def preprocess_tweet(text, model_type="embedding"):
	"""
	Preprocess a tweet based on the target model type.

	Parameters:
		text (str): The raw tweet text.
		model_type (str): One of ["embedding", "classic", "bertweet"].
			- "bertweet": minimal cleaning, tailored for BERTweet tokenizer
			- "embedding": general BERT/RoBERTa-style embedding models

	Returns:
		str: Preprocessed tweet text.
	"""
	# Fix text encoding issues
	text = html.unescape(text)
	text = unicodedata.normalize("NFKD", text)

	# Remove leading retweet marker "RT @user:"
	text = re.sub(r'^RT\s+@[\w_]+:\s+', '', text)

	# --- BERTweet-specific preprocessing ---
	if model_type == "bertweet":
		# DO NOT remove URLs, mentions, emojis
		# Just clean up retweet and encoding
		return text.strip()

	# --- For all other models ---
	# Replace URLs with placeholder
	text = re.sub(r'http\S+', '', text)
	
	# Replace mentions with placeholder
	text = re.sub(r'@\w+', '', text)

	# Convert emojis to text (e.g. 😢 → :crying_face:)
	text = emoji.demojize(text, delimiters=(" ", " "))

	# Remove '#' from hashtags, keep the word
	text = re.sub(r'#', '', text)

	# Normalize whitespace
	text = re.sub(r'\s+', ' ', text).strip()

	return text

In [233]:
# Apply preprocessing and create new columns for each model
df['cleaned_text_bert'] = df['tweet_text'].apply(lambda x: preprocess_tweet(x, model_type='embedding'))
df['cleaned_text_bertweet'] = df['tweet_text'].apply(lambda x: preprocess_tweet(x, model_type='bertweet'))


In [234]:
df[['tweet_text', 'cleaned_text_bert','cleaned_text_bertweet'] ].head(5)

,tweet_text,cleaned_text_bert,cleaned_text_bertweet
0,RT @Gizmodo: Wildfires raging through Northern California are terrifying https://t.co/dI73RFzX2i https://t.co/k4KnvIimsU,Wildfires raging through Northern California are terrifying,Wildfires raging through Northern California are terrifying https://t.co/dI73RFzX2i https://t.co/k4KnvIimsU
1,PHOTOS: Deadly wildfires rage in California https://t.co/td9xT3vXOL https://t.co/OimwAncLew,PHOTOS: Deadly wildfires rage in California,PHOTOS: Deadly wildfires rage in California https://t.co/td9xT3vXOL https://t.co/OimwAncLew
2,"RT @Cal_OES: PLS SHARE: We're capturing wildfire response, recovery info here: https://t.co/r89LKpjLPj https://t.co/HiA1oQF2Ax","PLS SHARE: We're capturing wildfire response, recovery info here:","PLS SHARE: We're capturing wildfire response, recovery info here: https://t.co/r89LKpjLPj https://t.co/HiA1oQF2Ax"
3,"RT @Cal_OES: PLS SHARE: We're capturing wildfire response, recovery info here: https://t.co/r89LKpjLPj https://t.co/HiA1oQF2Ax","PLS SHARE: We're capturing wildfire response, recovery info here:","PLS SHARE: We're capturing wildfire response, recovery info here: https://t.co/r89LKpjLPj https://t.co/HiA1oQF2Ax"
4,RT @TIME: California's raging wildfires as you've never seen them before https://t.co/OksQOZ2LHH https://t.co/oHTMbrM2Jx,California's raging wildfires as you've never seen them before,California's raging wildfires as you've never seen them before https://t.co/OksQOZ2LHH https://t.co/oHTMbrM2Jx


In [235]:
path_preprocessed = os.path.join(data_dir, "preprocessed_annotations.tsv")

df.to_csv(path_preprocessed, sep="\t", index=False)